# YouTube Buddy X — Wan 2.2 video pilot (free Kaggle GPU)

Generates real AI videos with **Wan 2.2 TI2V-5B** (Apache 2.0 licence — free for commercial use) on Kaggle's free T4 GPU.

**Before you click Run All** (right-hand panel → *Session options*):
1. **Accelerator:** `GPU T4 x2` (or `GPU P100`).
2. **Internet:** `On` (Kaggle asks for phone verification once).
3. Then **Run All**. First run downloads the model (~25 GB, about 5–10 min). Each clip then takes roughly 10–25 min on a T4 (estimate — the notebook measures it).

At the end it writes `pilot_report.json` and the MP4s to the **Output** tab. Send the report (or a screenshot of the last cell) back to Claude.

In [ ]:
# 1. Check the GPU
import subprocess, torch, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")
assert torch.cuda.is_available(), "No GPU — set Accelerator to 'GPU T4 x2' in Session options, then restart."
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {gpu} · {vram:.1f} GB · torch {torch.__version__} · python {sys.version.split()[0]}")

In [ ]:
# 2. Install the model library (Diffusers with Wan 2.2 support)
!pip install -q -U "diffusers>=0.35.1" "transformers>=4.49" accelerate ftfy imageio imageio-ffmpeg sentencepiece

In [ ]:
# 3. Settings — edit freely
MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"   # Apache 2.0
HEIGHT, WIDTH = 480, 832      # must be multiples of 32; 704x1280 (720p) is native but very slow / may not fit a T4
NUM_FRAMES = 81               # must be 4k+1; 81 frames @ 24 fps ≈ 3.4 s
FPS = 24
STEPS = 30                    # 50 = best quality, 30 = good balance, 20 = fast draft
GUIDANCE = 5.0
SEED = 42

NEGATIVE = ("blurry, low quality, distorted face, extra fingers, deformed hands, watermark, text, logo, "
            "static frame, jpeg artifacts, oversaturated")

PROMPTS = [
    ("diwali_family", "A cinematic shot of an Indian family lighting diyas on a balcony during Diwali at dusk, warm golden light, "
                      "fireworks bursting softly in the background, gentle camera push in, shallow depth of field"),
    ("goa_beach_vlog", "A young Indian travel vlogger walking along a Goa beach at golden hour holding a camera, waves rolling in, "
                       "palm trees swaying, handheld tracking shot, natural colours"),
    ("chai_closeup", "Close-up of masala chai being poured from a steel kettle into a glass cup at a Mumbai street stall, "
                     "steam rising, morning sunlight, slow motion, macro lens"),
]

In [ ]:
# 4. Load the model (fits a 16 GB T4 by keeping only the active part on the GPU)
import time, torch
from diffusers import WanPipeline, AutoencoderKLWan

# T4/P100 have no fast bfloat16, so the big transformer runs in float16; the VAE stays float32 for clean colours.
DTYPE = torch.float16 if torch.cuda.get_device_capability(0) < (8, 0) else torch.bfloat16
t0 = time.time()
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=DTYPE)
pipe.enable_model_cpu_offload()          # moves each component to the GPU only while it runs
try:
    pipe.vae.enable_tiling()             # decodes the video in tiles to save GPU memory
except Exception as e:
    print("VAE tiling not available:", e)
LOAD_S = round(time.time() - t0)
print(f"Loaded in {LOAD_S}s with transformer dtype {DTYPE}")

In [ ]:
# 5. Generate the clips
import json, os, numpy as np, torch
from diffusers.utils import export_to_video

def looks_broken(frames):
    a = np.asarray(frames, dtype=np.float32)
    return (not np.isfinite(a).all()) or a.std() < 0.01   # NaNs or a flat black/grey video

def generate(name, prompt, dtype_note):
    torch.cuda.reset_peak_memory_stats()
    g = torch.Generator(device="cpu").manual_seed(SEED)
    t = time.time()
    frames = pipe(prompt=prompt, negative_prompt=NEGATIVE, height=HEIGHT, width=WIDTH, num_frames=NUM_FRAMES,
                  guidance_scale=GUIDANCE, num_inference_steps=STEPS, generator=g).frames[0]
    secs = time.time() - t
    path = f"/kaggle/working/{name}.mp4"
    export_to_video(frames, path, fps=FPS)
    return {"name": name, "file": path, "seconds": round(secs), "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
            "broken": bool(looks_broken(frames)), "dtype": dtype_note}

results = []
for i, (name, prompt) in enumerate(PROMPTS):
    print(f"[{i+1}/{len(PROMPTS)}] {name} …")
    try:
        r = generate(name, prompt, str(DTYPE))
        if r["broken"] and DTYPE == torch.float16:
            print("  float16 gave a broken video — retrying this clip in bfloat16 (slower on this GPU)")
            pipe.transformer.to(torch.bfloat16)
            r = generate(name, prompt, "torch.bfloat16 (fallback)")
            pipe.transformer.to(DTYPE)
    except torch.cuda.OutOfMemoryError:
        r = {"name": name, "error": "out of GPU memory — lower HEIGHT/WIDTH or NUM_FRAMES in cell 3"}
        torch.cuda.empty_cache()
    except Exception as e:
        r = {"name": name, "error": f"{type(e).__name__}: {e}"[:500]}
    print("  ", r)
    results.append(r)

In [ ]:
# 6. Watch the results
from IPython.display import Video, display, HTML
for r in results:
    if "file" in r:
        display(HTML(f"<h4>{r['name']} — {r['seconds']} s to generate, peak GPU {r['peak_gpu_gb']} GB</h4>"))
        display(Video(r["file"], embed=True, width=640))
    else:
        display(HTML(f"<h4 style='color:#c00'>{r['name']} failed: {r['error']}</h4>"))

In [ ]:
# 7. Report for Claude — copy this output or download pilot_report.json from the Output tab
report = {"gpu": gpu, "vram_gb": round(vram, 1), "model": MODEL_ID, "resolution": f"{WIDTH}x{HEIGHT}", "frames": NUM_FRAMES,
          "fps": FPS, "steps": STEPS, "load_seconds": LOAD_S, "torch": torch.__version__,
          "diffusers": __import__("diffusers").__version__, "results": results}
json.dump(report, open("/kaggle/working/pilot_report.json", "w"), indent=2)
print(json.dumps(report, indent=2))